### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [70]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [71]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [72]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [73]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [74]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [75]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [76]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [77]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [78]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [79]:
# tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [80]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [81]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [82]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [83]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [84]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [85]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [86]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 1911, 1825, 1828], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [87]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [88]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [89]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [90]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](20,)","[480.,584.,591.,...,564.,465.,377.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](20,)","[-3.16,-2.96,-2.95,...,-3. ,-3.19,-3.4 ]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](20,)","[ 0, 1, 2,...,17,18,19]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](20, 101631)","[[0. ,0.94,0. ,...,0. ,0. ,0. ], [1.39,0.6 ,0. ,...,0. ,0. ,0. ], [0.95,0.14,0. ,...,0. ,0. ,0. ], ..., [0.42,2.9 ,0.04,...,0. ,0. ,0. ], [0.61,1.36,0. ,...,0. ,0. ,0. ], [0.03,0.38,0. ,...,0. ,0. ,0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](20, 101631)","[[-11.56,-10.9 ,-11.56,...,-11.56,-11.56,-11.56], [-10.69,-11.1 ,-11.56,...,-11.56,-11.56,-11.56], [-10.9 ,-11.44,-11.57,...,-11.57,-11.57,-11.57], ..., [-11.22,-10.21,-11.54,...,-11.57,-11.57,-11.57], [-11.09,-10.7 ,-11.56,...,-11.56,-11.56,-11.56], [-11.53,-11.23,-11.56,...,-11.56,-11.56,-11.56]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,101631


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [91]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [92]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


## 1. Similaridad de documentos

Tomé 5 documentos al azar del conjunto de entrenamiento (con semilla fija para que el resultado sea reproducible) y, para cada uno, busqué los 5 documentos más similares según la similaridad coseno sobre la vectorización TF-IDF ya ajustada (`X_train`).

In [93]:
rng = np.random.default_rng(42)
random_idxs = rng.choice(X_train.shape[0], 5, replace=False)
print(random_idxs)

[8754 4965 7404 1009 4899]


In [94]:
def mostrar_similares(idx, n=5, chars=300):
    cossim = cosine_similarity(X_train[idx], X_train)[0]
    mostsim = np.argsort(cossim)[::-1][1:n+1]
    print('=' * 100)
    print(f'DOCUMENTO {idx} | clase: {newsgroups_train.target_names[y_train[idx]]}')
    print('-' * 100)
    print(newsgroups_train.data[idx].strip()[:chars])
    print('-' * 100)
    for i in mostsim:
        texto = ' '.join(newsgroups_train.data[i].split())[:150]
        print(f'  [{i:5d}] sim={cossim[i]:.3f} | {newsgroups_train.target_names[y_train[i]]:<26} | {texto}')
    return y_train[mostsim]

coincidencias = 0
for idx in random_idxs:
    clases_vecinos = mostrar_similares(idx)
    coincidencias += (clases_vecinos == y_train[idx]).sum()
print(f'\nVecinos con la misma clase que el documento consultado: {coincidencias}/25')

DOCUMENTO 8754 | clase: talk.religion.misc
----------------------------------------------------------------------------------------------------
/(hudson)
/If someone inflicts pain on themselves, whether they enjoy it or not, they
/are hurting themselves.  They may be permanently damaging their body.

That is true.  It is also none of your business.  

Some people may also reason that by reading the bible and being a Xtian
you are permanentl
----------------------------------------------------------------------------------------------------
  [ 6552] sim=0.490 | talk.religion.misc         | If I have a habit that I really want to break, and I am willing to make whatever sacrifice I need to make to break it, then I do so. There have been b
  [10613] sim=0.481 | talk.religion.misc         | /(hudson) /Yes you do. Who is to say that it is immoral for onesself to experience /pain or to be hurt in some other way. Maybe unpleasant, but that d
  [ 3616] sim=0.465 | talk.religion.misc         |

**Análisis de los resultados**

* **Doc 8754 (`talk.religion.misc`)**: es una discusión sobre moral y autolesión con referencias a la Biblia ("Xtian"). 4 de los 5 vecinos son de la misma clase y el doc 10613 parece ser del mismo hilo (cita al mismo usuario, "/(hudson)"). El único vecino de otra clase (`talk.politics.mideast`) comparte el estilo de debate con citas anidadas y vocabulario argumentativo general. Las similaridades son relativamente altas (~0.46–0.49) porque es un texto largo con mucho vocabulario en común con su hilo.
* **Doc 4965 (`comp.sys.mac.hardware`)**: es un texto muy corto ("Plug the printer in the printer port, and the modem in the modem port"). Todos sus vecinos hablan de puertos, impresoras y módems, así que **por contenido** la similaridad tiene mucho sentido. Sin embargo, solo 2 de los 5 son de la misma clase: los demás son de `comp.sys.ibm.pc.hardware` y `comp.graphics`. Las clases de hardware de Mac y de PC comparten casi todo el vocabulario técnico y TF-IDF no puede distinguirlas en un mensaje tan breve.
* **Doc 7404 (`comp.os.ms-windows.misc`)**: pregunta cómo minimizar el Program Manager al abrir una aplicación. Ningún vecino es de su clase. Tres son de `comp.windows.x` (comparten "application"), y dos son documentos que contienen únicamente la palabra **"Hello,"** (de `sci.med` y `comp.graphics`). Esto muestra una limitación clara de la bolsa de palabras: en documentos cortos, una sola palabra genérica coincidente basta para dar una similaridad alta, sin relación semántica real.
* **Doc 1009 (`talk.politics.guns`)**: es un debate sobre el derecho a portar armas y la religión. 4 de los 5 vecinos son de la misma clase. El quinto es de `alt.atheism`, lo cual tiene sentido porque el texto discute "RELIGION" y "illogical dogma". Las similaridades son bajas (~0.14), porque el texto contiene muchos nombres propios e iniciales ("rl>", "kr>") que no aportan.
* **Doc 4899 (`sci.crypt`)**: aunque está publicado en el grupo de criptografía, su contenido es político (Noam Chomsky, medios, control de la información). Sus vecinos son 3 de `sci.crypt`, 1 de `talk.politics.mideast` y 1 de `alt.atheism`: todos debates o "flames" con vocabulario argumentativo. Este caso muestra que la etiqueta corresponde al grupo donde se publicó y no necesariamente al tema del texto.

**Conclusión**: solo 13 de los 25 vecinos (52%) comparten la clase del documento consultado. Cuando el documento es largo y de un tema específico (religión, armas), la similaridad coseno sobre TF-IDF encuentra documentos claramente relacionados. Falla en tres situaciones: (a) documentos muy cortos, donde una palabra genérica domina el vector; (b) clases temáticamente muy cercanas (hardware Mac vs PC, Windows vs X Window); (c) documentos cuyo contenido no coincide con el grupo donde fueron publicados. En muchos de los "errores" la similaridad sí tiene sentido por contenido, aunque no por etiqueta.

## 2. Clasificador por prototipos (zero-shot / 1-NN)

Cada documento de test se compara con **todos** los documentos de entrenamiento y se le asigna la etiqueta del documento de entrenamiento más similar (vecino más cercano con similaridad coseno). No hay entrenamiento propiamente dicho: el "modelo" es el conjunto de entrenamiento vectorizado.

In [95]:
# matriz de similaridad (n_test x n_train)
sim_test_train = cosine_similarity(X_test, X_train)
y_pred_proto = y_train[sim_test_train.argmax(axis=1)]

print(f'F1-score macro (prototipos): {f1_score(y_test, y_pred_proto, average="macro"):.4f}')
print(f'Accuracy (prototipos):       {(y_pred_proto == y_test).mean():.4f}')
print(f'F1-score macro (MultinomialNB base): {f1_score(y_test, y_pred, average="macro"):.4f}')

F1-score macro (prototipos): 0.5050
Accuracy (prototipos):       0.5089
F1-score macro (MultinomialNB base): 0.5854


Algunos documentos de test quedan vacíos tras quitar headers, footers y quotes, o solo contienen palabras que no están en el vocabulario de train. Su vector es nulo, la similaridad con todo es 0 y `argmax` devuelve siempre el índice 0. Vi cuántos son y cómo cambia la métrica sin ellos:

In [96]:
vacios = X_test.getnnz(axis=1) == 0
print(f'Documentos de test con vector nulo: {vacios.sum()} de {X_test.shape[0]}')
print(f'F1-score macro (prototipos, sin vacíos): {f1_score(y_test[~vacios], y_pred_proto[~vacios], average="macro"):.4f}')

Documentos de test con vector nulo: 224 de 7532
F1-score macro (prototipos, sin vacíos): 0.5171


In [97]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_proto, target_names=newsgroups_test.target_names))

                          precision    recall  f1-score   support

             alt.atheism       0.37      0.51      0.43       319
           comp.graphics       0.54      0.48      0.51       389
 comp.os.ms-windows.misc       0.51      0.46      0.48       394
comp.sys.ibm.pc.hardware       0.52      0.52      0.52       392
   comp.sys.mac.hardware       0.53      0.50      0.52       385
          comp.windows.x       0.70      0.59      0.64       395
            misc.forsale       0.63      0.46      0.53       390
               rec.autos       0.41      0.58      0.48       396
         rec.motorcycles       0.63      0.52      0.57       398
      rec.sport.baseball       0.65      0.54      0.59       397
        rec.sport.hockey       0.75      0.72      0.73       399
               sci.crypt       0.55      0.59      0.57       396
         sci.electronics       0.53      0.33      0.41       393
                 sci.med       0.65      0.49      0.56       396
         

**Análisis**

* El clasificador por prototipos obtiene un F1 macro de **~0.50**, por debajo del Naïve Bayes base (**~0.585**). El resultado es coherente con el punto 1, donde solo el 52% de los vecinos compartía la clase.
* Decidir con **un único vecino** hace al clasificador muy sensible al ruido: alcanza con que un documento corto o genérico de otra clase sea el más parecido para equivocarse. Naïve Bayes, en cambio, agrega estadísticas de **todos** los documentos de cada clase y por eso es más robusto.
* Los documentos vacíos se clasifican siempre con la clase del documento 0 de train. Esto penaliza un poco la métrica; al excluirlos, el F1 mejora levemente.
* En el reporte por clase, las clases con vocabulario muy específico tienen mejor F1: `rec.sport.hockey` (0.73) y `comp.windows.x` (0.64). Las peores son las que se solapan temáticamente con otras: `talk.religion.misc` (0.28), `talk.politics.misc` (0.31), `sci.electronics` (0.41) y `alt.atheism` (0.43). Las clases de religión se confunden entre sí y las de política también.
* Además, el método es costoso en inferencia: requiere una matriz de similaridad de 7532 × 11314 y escala linealmente con el tamaño del conjunto de entrenamiento.

## 3. Optimización de Naïve Bayes (MultinomialNB y ComplementNB)

Exploré combinaciones de parámetros del vectorizador TF-IDF (sin modificar `ngram_range`) y del modelo:

* **Vectorizador**: `stop_words`, `sublinear_tf`, `max_df`, `min_df`, `use_idf`, `norm`.
* **Modelo**: tipo (`MultinomialNB` / `ComplementNB`), suavizado `alpha` y, en MultinomialNB, `fit_prior`.

Siguiendo la consigna, la métrica a maximizar es el F1 macro en test. En la práctica esto debería hacerse con validación cruzada sobre train, porque elegir hiperparámetros mirando test da una estimación optimista.

In [98]:
import pandas as pd

vect_configs = {
    'default': {},
    'stop_words': dict(stop_words='english'),
    'sublinear_tf': dict(sublinear_tf=True),
    'max_df=0.3': dict(max_df=0.3),
    'min_df=2': dict(min_df=2),
    'stop+sublinear+min_df=2': dict(stop_words='english', sublinear_tf=True, min_df=2),
    'use_idf=False': dict(use_idf=False),
    'norm=None': dict(norm=None),
}
model_configs = {
    'MultinomialNB': lambda a: MultinomialNB(alpha=a),
    'MultinomialNB (fit_prior=False)': lambda a: MultinomialNB(alpha=a, fit_prior=False),
    'ComplementNB': lambda a: ComplementNB(alpha=a),
}
alphas = [1.0, 0.3, 0.1, 0.03, 0.01]

resultados = []
for vname, vparams in vect_configs.items():
    vect = TfidfVectorizer(**vparams)
    Xtr = vect.fit_transform(newsgroups_train.data)
    Xte = vect.transform(newsgroups_test.data)
    for mname, make_model in model_configs.items():
        for a in alphas:
            pred = make_model(a).fit(Xtr, y_train).predict(Xte)
            resultados.append(dict(vectorizador=vname, modelo=mname, alpha=a,
                                   f1_macro=f1_score(y_test, pred, average='macro')))

df_res = pd.DataFrame(resultados)
df_res.sort_values('f1_macro', ascending=False).head(10)

,vectorizador,modelo,alpha,f1_macro
56,max_df=0.3,ComplementNB,0.30,0.700296
11,default,ComplementNB,0.30,0.699937
26,stop_words,ComplementNB,0.30,0.699312
103,use_idf=False,ComplementNB,0.03,0.699230
41,sublinear_tf,ComplementNB,0.30,0.698983
102,use_idf=False,ComplementNB,0.10,0.698741
71,min_df=2,ComplementNB,0.30,0.697596
12,default,ComplementNB,0.10,0.695365
57,max_df=0.3,ComplementNB,0.10,0.694385
25,stop_words,ComplementNB,1.00,0.693611


Mejor F1 obtenido por cada combinación vectorizador/modelo (tomando el mejor `alpha`):

In [99]:
df_res.pivot_table(index='vectorizador', columns='modelo', values='f1_macro', aggfunc='max').round(4)

modelo,ComplementNB,MultinomialNB,MultinomialNB (fit_prior=False)
vectorizador,,,
default,0.6999,0.6829,0.6895
max_df=0.3,0.7003,0.6828,0.6897
min_df=2,0.6976,0.6810,0.6880
norm=None,0.6261,0.6321,0.6361
stop+sublinear+min_df=2,0.6930,0.6777,0.6875
stop_words,0.6993,0.6844,0.6887
sublinear_tf,0.6990,0.6787,0.6839
use_idf=False,0.6992,0.6653,0.6772


Efecto de `alpha` con el vectorizador por defecto:

In [100]:
df_res[df_res.vectorizador == 'default'].pivot_table(index='alpha', columns='modelo', values='f1_macro').round(4)

modelo,ComplementNB,MultinomialNB,MultinomialNB (fit_prior=False)
alpha,,,
0.01,0.6689,0.6829,0.6895
0.03,0.6810,0.6767,0.6842
0.10,0.6954,0.6565,0.6664
0.30,0.6999,0.6322,0.6429
1.00,0.6930,0.5854,0.6013


Entrené el mejor modelo encontrado y vi el detalle por clase:

In [101]:
best = df_res.loc[df_res.f1_macro.idxmax()]
print(best)

best_vect = TfidfVectorizer(**vect_configs[best.vectorizador])
Xtr_best = best_vect.fit_transform(newsgroups_train.data)
Xte_best = best_vect.transform(newsgroups_test.data)
best_clf = model_configs[best.modelo](best.alpha).fit(Xtr_best, y_train)
y_pred_best = best_clf.predict(Xte_best)
print(f'\nF1-score macro: {f1_score(y_test, y_pred_best, average="macro"):.4f}\n')
print(classification_report(y_test, y_pred_best, target_names=newsgroups_test.target_names))

vectorizador      max_df=0.3
modelo          ComplementNB
alpha                    0.3
f1_macro            0.700296
Name: 56, dtype: object

F1-score macro: 0.7003

                          precision    recall  f1-score   support

             alt.atheism       0.33      0.45      0.38       319
           comp.graphics       0.72      0.72      0.72       389
 comp.os.ms-windows.misc       0.72      0.59      0.65       394
comp.sys.ibm.pc.hardware       0.63      0.70      0.66       392
   comp.sys.mac.hardware       0.78      0.72      0.75       385
          comp.windows.x       0.82      0.79      0.80       395
            misc.forsale       0.74      0.73      0.74       390
               rec.autos       0.80      0.75      0.77       396
         rec.motorcycles       0.82      0.78      0.80       398
      rec.sport.baseball       0.92      0.84      0.88       397
        rec.sport.hockey       0.87      0.95      0.91       399
               sci.crypt       0.76      0

**Análisis**

* **`alpha` es el hiperparámetro más influyente.** Con el valor por defecto (`alpha=1`), `MultinomialNB` obtiene F1 ≈ 0.585. Al bajar a `alpha≈0.01` sube a ≈ 0.68. Con valores de TF-IDF normalizados (todos < 1), un `alpha=1` es un suavizado enorme en comparación con los "conteos" reales y aplana las distribuciones de cada clase. Reducirlo deja que las palabras discriminativas pesen más.
* **ComplementNB supera a MultinomialNB** en casi todas las configuraciones (mejor F1 ≈ 0.70). ComplementNB estima los parámetros de cada clase a partir de los documentos de **todas las demás clases** (el complemento). Así tiene más datos por estimación y es menos sensible al desbalance, lo que favorece justamente al F1 macro. Además es mucho menos sensible a `alpha`: su óptimo está en valores más altos (0.1–0.3).
* `fit_prior=False` (prior uniforme) mejora levemente a MultinomialNB, por el mismo motivo: no favorece a las clases mayoritarias.
* **Parámetros del vectorizador**: con el `alpha` óptimo, las diferencias entre configuraciones son de pocas milésimas. Quitar stop words o filtrar términos muy frecuentes (`max_df=0.3`) elimina ruido, pero ese ruido ya pesa poco en TF-IDF gracias al IDF. La mejor configuración fue `max_df=0.3` + ComplementNB con `alpha=0.3` (F1 = 0.7003). `max_df`, `min_df` y `sublinear_tf` casi no cambian el resultado. `norm=None` empeora claramente: sin normalizar, los documentos largos dominan las estadísticas de su clase.
* El rendimiento se estabiliza en **F1 macro ≈ 0.70**. Sin cambiar `ngram_range`, los modelos de bolsa de palabras no pueden capturar orden ni contexto, y las clases temáticamente superpuestas siguen siendo la principal fuente de error. En el reporte por clase, `talk.religion.misc` (F1 0.25, recall 0.16) es absorbida por `soc.religion.christian` y `alt.atheism`, y `talk.politics.misc` (0.52) se confunde con las demás clases de política. Los deportes y `comp.windows.x` superan 0.80.
* La mejora respecto del modelo base (0.585 → ~0.70) y del clasificador por prototipos (~0.50) es significativa.

## 4. Similaridad entre palabras (matriz término-documento)

Transpuse la matriz documento-término `X_train` (11314 × 101631) para obtener la matriz término-documento (101631 × 11314). Ahora cada **palabra** se representa con un vector de dimensión igual a la cantidad de documentos, cuyos valores son su TF-IDF en cada documento. Dos palabras son similares si aparecen (con peso alto) en los mismos documentos. Es decir, a cada palabra la define el conjunto de documentos en los que aparece, sin importar qué palabras tiene cerca dentro del texto.

Palabras elegidas manualmente, una por cada temática bien diferenciada del dataset: **car, god, space, gun, windows**.

In [102]:
X_train_T = X_train.T.tocsr()
print(f'Matriz término-documento: {X_train_T.shape}')

def palabras_similares(palabra, XT, vectorizer, idx2w, n=5):
    i = vectorizer.vocabulary_[palabra]
    cossim = cosine_similarity(XT[i], XT)[0]
    top = np.argsort(cossim)[::-1][1:n+1]
    return [(idx2w[j], round(cossim[j], 3)) for j in top]

palabras = ['car', 'god', 'space', 'gun', 'windows']
for p in palabras:
    print(f'{p:<8} -> {palabras_similares(p, X_train_T, tfidfvect, idx2word)}')

Matriz término-documento: (101631, 11314)
car      -> [('cars', np.float64(0.18)), ('criterium', np.float64(0.177)), ('civic', np.float64(0.175)), ('owner', np.float64(0.169)), ('dealer', np.float64(0.168))]
god      -> [('jesus', np.float64(0.269)), ('bible', np.float64(0.262)), ('that', np.float64(0.256)), ('existence', np.float64(0.255)), ('christ', np.float64(0.251))]
space    -> [('nasa', np.float64(0.33)), ('seds', np.float64(0.297)), ('shuttle', np.float64(0.293)), ('enfant', np.float64(0.28)), ('seti', np.float64(0.246))]
gun      -> [('guns', np.float64(0.358)), ('crime', np.float64(0.244)), ('handgun', np.float64(0.239)), ('homicides', np.float64(0.233)), ('firearms', np.float64(0.233))]
windows  -> [('dos', np.float64(0.304)), ('ms', np.float64(0.232)), ('microsoft', np.float64(0.222)), ('nt', np.float64(0.214)), ('for', np.float64(0.193))]


En algunos casos aparecen stop words ("that", "for") entre los vecinos: son palabras tan frecuentes que coinciden con casi cualquier término. Repetí el experimento con un vectorizador que las elimina:

In [103]:
tfidf_sw = TfidfVectorizer(stop_words='english')
X_train_T_sw = tfidf_sw.fit_transform(newsgroups_train.data).T.tocsr()
idx2word_sw = {v: k for k, v in tfidf_sw.vocabulary_.items()}

for p in palabras:
    print(f'{p:<8} -> {palabras_similares(p, X_train_T_sw, tfidf_sw, idx2word_sw)}')

car      -> [('cars', np.float64(0.19)), ('criterium', np.float64(0.173)), ('dealer', np.float64(0.173)), ('civic', np.float64(0.171)), ('owner', np.float64(0.164))]
god      -> [('jesus', np.float64(0.281)), ('bible', np.float64(0.276)), ('christ', np.float64(0.267)), ('faith', np.float64(0.259)), ('existence', np.float64(0.259))]
space    -> [('nasa', np.float64(0.328)), ('shuttle', np.float64(0.29)), ('seds', np.float64(0.285)), ('enfant', np.float64(0.269)), ('exploration', np.float64(0.24))]
gun      -> [('guns', np.float64(0.377)), ('crime', np.float64(0.253)), ('handgun', np.float64(0.245)), ('firearms', np.float64(0.24)), ('control', np.float64(0.232))]
windows  -> [('dos', np.float64(0.304)), ('ms', np.float64(0.232)), ('microsoft', np.float64(0.221)), ('nt', np.float64(0.214)), ('file', np.float64(0.192))]


**Análisis**

* **car**: devuelve "cars" (plural), "dealer", "owner" y "civic" (Honda Civic), todos del dominio automotor (`rec.autos`). "criterium" es probablemente un término raro que aparece en pocos documentos sobre autos. Con tan pocas apariciones, su vector queda concentrado en esos documentos y la similaridad resulta alta por casualidad.
* **god**: "jesus", "bible", "christ", "existence" corresponden a las discusiones religiosas y ateas (`alt.atheism`, `soc.religion.christian`). "existence" refleja el debate sobre la existencia de Dios, propio de `alt.atheism`.
* **space**: "nasa", "shuttle", "seds" (Students for the Exploration and Development of Space) y "seti" son vecinos muy coherentes con `sci.space`. "enfant" probablemente viene de "L'Enfant Plaza", la dirección de oficinas de NASA en Washington que aparece en comunicados de prensa repetidos: es co-ocurrencia de formato, no semántica.
* **gun**: "guns", "handgun", "firearms", "crime", "homicides". Capta tanto los sinónimos como el contexto del debate sobre control de armas (`talk.politics.guns`).
* **windows**: "dos", "ms", "microsoft", "nt" (Windows NT): todos del ecosistema Microsoft (`comp.os.ms-windows.misc`).

**Conclusiones**

* Las similaridades capturan sobre todo **relaciones temáticas o de co-ocurrencia** ("space" ↔ "nasa", "gun" ↔ "crime"), no sinonimia estricta. Dos palabras son similares si se usan en los mismos documentos, aunque no signifiquen lo mismo.
* La vectorización no tiene información morfológica: "car" / "cars" y "gun" / "guns" son términos distintos que el modelo relaciona solo porque co-ocurren. Un stemming o lematización los unificaría.
* Las palabras muy frecuentes (stop words) y las muy raras (que aparecen en 1 o 2 documentos) generan vecinos poco interpretables. Al quitar las stop words, "that" y "for" se reemplazan por términos con sentido: "faith" para *god*, "file" para *windows*. Además aparecen "exploration" para *space* y "control" para *gun* (por *gun control*).
* Las similaridades absolutas son bajas (0.2–0.35), porque los vectores tienen dimensión ~11 000 y son muy dispersos. Aun así, el ranking es claramente interpretable. Es una versión primitiva de los *word embeddings*: métodos como Word2Vec usan la misma hipótesis de distribución, pero con contextos locales (ventanas de palabras) y vectores densos de baja dimensión.